In [1]:
import pandas as pd
import numpy as np

# 1. Load data
file_path = 'data_raw/Sample - Superstore.csv'
try:
    df_raw = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df_raw = pd.read_csv(file_path, encoding='windows-1252')

print(f"Loaded successfully! Initial rows: {len(df_raw)}")

# Standardize column names to lowercase and strip spaces
df_raw.columns = df_raw.columns.str.strip().str.lower()

# 2. Basic cleaning
df_clean = df_raw.drop_duplicates().copy()
df_clean['postal code'] = df_clean['postal code'].fillna('00000').astype(str)
df_clean['order date'] = pd.to_datetime(df_clean['order date'])
df_clean['ship date'] = pd.to_datetime(df_clean['ship date'])
df_clean['state'] = df_clean['state'].str.strip()
df_clean['city'] = df_clean['city'].str.strip()

# 3. Create the 4 Relational Tables cleanly
# Customers Table
customers = df_clean[['customer id', 'customer name', 'segment']].drop_duplicates()
customers.columns = ['customer_id', 'customer_name', 'segment']

# Products Table (handling sub-category name variation)
subcat_col = 'sub-category' if 'sub-category' in df_clean.columns else 'sub category'
products = df_clean[['product id', 'category', subcat_col, 'product name']].drop_duplicates(subset=['product id'])
products.columns = ['product_id', 'category', 'sub_category', 'product_name']

# Regions Table (using state and city to avoid missing key errors)
regions = df_clean[['state', 'city', 'postal code']].drop_duplicates().reset_index(drop=True)
regions['region_id'] = regions.index + 1
regions.columns = ['state', 'city', 'postal_code', 'region_id']

# Merge regions back safely using state, city, and postal code
df_orders = df_clean.merge(regions, left_on=['state', 'city', 'postal code'], right_on=['state', 'city', 'postal_code'], how='left')

# Orders Table
orders = df_orders[['row id', 'order id', 'order date', 'ship date', 'ship mode', 
                    'customer id', 'product id', 'region_id', 'sales', 'quantity', 'discount', 'profit']]
orders.columns = ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 
                  'customer_id', 'product_id', 'region_id', 'sales', 'quantity', 'discount', 'profit']

# 4. Save the 4 CSV files
customers.to_csv('customers.csv', index=False)
products.to_csv('products.csv', index=False)
regions.to_csv('regions.csv', index=False)
orders.to_csv('orders.csv', index=False)

print("✔ Success! Generated: customers.csv, products.csv, regions.csv, and orders.csv")

Loaded successfully! Initial rows: 9994
✔ Success! Generated: customers.csv, products.csv, regions.csv, and orders.csv
